> **[MOD 2026-09-13 HNK] Copia de `PIPELINE_2026_all_in_one.ipynb` para una corrida aparte: solo el tile 20HNK, junio-julio 2026.** El pipeline es idéntico; solo cambian CONFIG, el sufijo de los mosaicos (Etapa 3) y la celda final de selección.

# Pipeline 2026 — TODO EN UNO (descarga → composite → merge → clip → tile → INBNV)

Genera tiles nuevos de **2026** y su GT semiautomático (polígonos INBNV), replicando
exactamente tu pipeline 01-06, pero guardando **todo bajo una carpeta nueva** (`BASE_2026`)
para no mezclar con lo ya hecho.

**Flujo:** corré las celdas en orden. Cada etapa lee de la carpeta de la etapa anterior.
La nomenclatura de tiles queda idéntica: `2026_<mes>_<dia>_<seq>_tile_rNN_cNN.tif`.

**Lo único que tocás es la celda de CONFIG** (fechas, tiles MGRS, carpeta raíz).

## 0. Configuración (editá solo esto)

In [1]:
import os
import re
import glob
from pathlib import Path
from datetime import date, datetime
from collections import defaultdict
import numpy as np
import rasterio
from tqdm import tqdm

# ===================== CONFIG 2026 (LO ÚNICO QUE TOCÁS) =====================
# Carpeta raíz NUEVA: todo lo de 2026 vive acá, separado de lo anterior
# [MOD 2026-09-13 HNK] carpeta propia para esta corrida (no toca D:\Silos\Base_2026)
BASE_2026 = Path(r"D:\Silos\Base_2026_HNK_junjul")

# --- Descarga (AWS S3 Sentinel-2 L2A) ---
# Tiles MGRS que cubren tu zona (Marcos Juárez). Confirmá que sean los mismos
# que usaste para 2020-2025. Si tu zona cae en un solo tile, dejá uno solo.
# [MOD 2026-09-13 HNK] solo 20HNK (el jun/jul original se armó con 20HNH + 20HNJ)
MGRS_TILES = ["20HNK"]
# [MOD 2026-09-13 HNK] mismas fechas que el jun/jul original (última escena usada: 24/07)
START_DATE = date(2026, 6, 1)
END_DATE   = date(2026, 7, 24)
# [MOD 2026-09-13 HNK] meses que procesan las etapas 2, 5 y 6
MESES_A_PROCESAR = [6, 7]
# [MOD 2026-09-13 HNK] NUEVO: sufijo en mosaicos y teselas para no chocar con los nombres del jun/jul original
SUFIJO = "_HNK"

# --- Clip a zona agrícola (mismo shapefile que siempre) ---
SHAPEFILE_AGRI = r"C:\Users\m\Documents\Tesis_Silos\Silos_FR_MCDtf\GeoDatos\argentina_Agriculture_MERGE_Marcos_Juarez.shp"

# --- Tiling (idéntico a tu 05) ---
TILE_SIZE = 1024
OVERLAP_PERCENT = 20

# --- INBNV / GT (idéntico a tu 06) ---
BANDA_ROJA = 1      # B04
BANDA_VERDE = 2     # B03
BANDA_NIR = 4       # B08
PERCENTILE = 99.8
MIN_PIXELS = 3
BUFFER_PIXELS = 1
# ===========================================================================

# Subcarpetas derivadas (NO tocar) — replican tu estructura, pero bajo BASE_2026
DL_DIR        = BASE_2026 / "s2_aws_downloads_structured"
COMPOSITE_DIR = BASE_2026 / "procesado_sin_nubes"
MOSAIC_DIR    = COMPOSITE_DIR / "mosaics"
CLIP_DIR      = MOSAIC_DIR / "mosaics_clipped"
TILES_DIR     = COMPOSITE_DIR / "tiles"
INDEX_DIR     = TILES_DIR / "indices_INBNV"
MASK_DIR      = INDEX_DIR / "masks"
POLY_DIR      = MASK_DIR / "polygons"

# [MOD 2026-09-13] NUEVO: helper para filtrar por mes (nombres tipo 2026_8_1_0_...)
def mes_de(nombre):
    m = re.search(r'\d{4}_(\d{1,2})_\d{1,2}_', os.path.basename(str(nombre)))
    return int(m.group(1)) if m else None

for d in [BASE_2026, DL_DIR, COMPOSITE_DIR, MOSAIC_DIR, CLIP_DIR, TILES_DIR, INDEX_DIR, MASK_DIR, POLY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Carpeta raíz 2026:", BASE_2026)
print("Rango de fechas:", START_DATE, "->", END_DATE)
print("Tiles MGRS:", MGRS_TILES)
print("Estructura creada. Tiles finales irán a:", TILES_DIR)


Carpeta raíz 2026: D:\Silos\Base_2026_HNK_junjul
Rango de fechas: 2026-06-01 -> 2026-07-24
Tiles MGRS: ['20HNK']
Estructura creada. Tiles finales irán a: D:\Silos\Base_2026_HNK_junjul\procesado_sin_nubes\tiles


## 1. Descarga (AWS S3)
Baja las bandas Sentinel-2 L2A para los tiles MGRS y el rango de fechas, manteniendo la
estructura de carpetas que espera la etapa de composite.

In [2]:
# ============ ETAPA 1/6: DESCARGA (AWS S3) — SOLO bandas necesarias ============
# Requiere: pip install boto3
import warnings, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "sentinel-s2-l2a"
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# Lo único que necesita el composite (tu etapa 2): 4 bandas en R10m + SCL en R20m
NEEDED_SUFFIXES = (
    "/R10m/B02.jp2", "/R10m/B03.jp2", "/R10m/B04.jp2", "/R10m/B08.jp2",
    "/R20m/SCL.jp2",
)

def tile_to_prefix(tile_code):
    return f"tiles/{tile_code[:2]}/{tile_code[2]}/{tile_code[3:]}/"

total_descargados = 0
for tile in MGRS_TILES:
    prefix = tile_to_prefix(tile)
    print(f"\nTile {tile} -> {prefix}")
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET, Prefix=prefix, RequestPayer="requester")

    keys = []
    for page in pages:
        for obj in page.get("Contents", []):
            key = obj["Key"]
            parts = key.split("/")  # tiles/20/H/NH/YYYY/M/D/SEQ/...
            try:
                yyyy = int(parts[4]); mm = int(parts[5]); dd = int(parts[6])
                dt = datetime(yyyy, mm, dd).date()
            except Exception:
                continue
            if not (START_DATE <= dt <= END_DATE):
                continue
            if key.endswith(NEEDED_SUFFIXES):   # <-- solo los 5 archivos útiles
                keys.append(key)

    print(f"  Archivos necesarios (5 por escena): {len(keys)}")
    for key in tqdm(keys, desc=f"Descargando {tile}"):
        local_path = DL_DIR / key
        local_path.parent.mkdir(parents=True, exist_ok=True)
        if local_path.exists():
            continue
        try:
            s3.download_file(BUCKET, key, str(local_path), ExtraArgs={"RequestPayer": "requester"})
            total_descargados += 1
        except Exception as e:
            print(f"  Error descargando {key}: {e}")

print(f"\nDescarga finalizada. Archivos nuevos: {total_descargados}")
print(f"~{total_descargados // 5} escenas (5 archivos c/u). Guardado en: {DL_DIR}")


Tile 20HNK -> tiles/20/H/NK/
  Archivos necesarios (5 por escena): 150


Descargando 20HNK: 100%|##########| 150/150 [40:55<00:00, 16.37s/it]



Descarga finalizada. Archivos nuevos: 150
~30 escenas (5 archivos c/u). Guardado en: D:\Silos\Base_2026_HNK_junjul\s2_aws_downloads_structured


## 2. Composite 4 bandas sin nubes (R,G,B,NIR con máscara SCL)

In [3]:
# ============ ETAPA 2/6: COMPOSITE 4 BANDAS SIN NUBES (tu 02) ============
from rasterio.enums import Resampling
from rasterio import warp

def procesar_imagenes_sentinel(ruta_base, ruta_salida):
    os.makedirs(ruta_salida, exist_ok=True)
    print(f"Composite -> {ruta_salida}")

    valores_scl_validos = [4, 5, 6, 7, 11]
    bandas_10m_a_usar = ['B04', 'B03', 'B02', 'B08']   # orden de salida: R, G, B, NIR

    n_ok = 0
    for root, dirs, files in os.walk(ruta_base):
        if root.endswith("R10m"):
            # [MOD 2026-09-13] solo fechas de MESES_A_PROCESAR (evita rehacer los composites de jun/jul)
            m_mes = re.search(r'[\\/]\d{4}[\\/](\d{1,2})[\\/]\d{1,2}[\\/]', root)
            if not m_mes or int(m_mes.group(1)) not in MESES_A_PROCESAR:
                continue
            ruta_scl_20m = root.replace("R10m", "R20m")
            archivo_scl = os.path.join(ruta_scl_20m, "SCL.jp2")
            rutas_bandas = {b: os.path.join(root, f"{b}.jp2") for b in bandas_10m_a_usar}

            if not os.path.exists(archivo_scl):
                continue
            if not all(os.path.exists(p) for p in rutas_bandas.values()):
                continue

            try:
                with rasterio.open(archivo_scl) as src_scl:
                    scl_data = src_scl.read(1)
                    perfil_scl = src_scl.profile
                    mascara_20m = np.isin(scl_data, valores_scl_validos)

                with rasterio.open(rutas_bandas['B04']) as src_10m_sample:
                    perfil_10m = src_10m_sample.profile
                    forma_destino = src_10m_sample.shape

                mascara_10m = np.empty(forma_destino, dtype=np.uint8)
                warp.reproject(
                    source=mascara_20m.astype(np.uint8),
                    destination=mascara_10m,
                    src_transform=perfil_scl['transform'], src_crs=perfil_scl['crs'],
                    dst_transform=perfil_10m['transform'], dst_crs=perfil_10m['crs'],
                    resampling=Resampling.nearest
                )
                mascara_10m_bool = mascara_10m.astype(bool)

                stack_10m = []
                for banda in bandas_10m_a_usar:
                    with rasterio.open(rutas_bandas[banda]) as src_banda:
                        stack_10m.append(src_banda.read(1))
                array_stack = np.array(stack_10m, dtype=perfil_10m['dtype'])

                valor_nodata = 0
                array_stack[:, ~mascara_10m_bool] = valor_nodata

                partes_ruta = root.replace(str(ruta_base), '').strip(os.sep).split(os.sep)
                nombre_archivo = f"{'_'.join(partes_ruta)}.tif"
                ruta_archivo_salida = os.path.join(ruta_salida, nombre_archivo)

                perfil_salida = perfil_10m.copy()
                perfil_salida.update({'driver': 'GTiff', 'count': len(bandas_10m_a_usar),
                                      'nodata': valor_nodata, 'compress': 'lzw'})

                with rasterio.open(ruta_archivo_salida, 'w', **perfil_salida) as dst:
                    dst.write(array_stack)
                n_ok += 1
            except Exception as e:
                print(f"  ERROR en {root}: {e}")

    print(f"Composites generados: {n_ok}")

procesar_imagenes_sentinel(str(DL_DIR / "tiles"), str(COMPOSITE_DIR))
print("--- Etapa 2 lista ---")


Composite -> D:\Silos\Base_2026_HNK_junjul\procesado_sin_nubes
Composites generados: 30
--- Etapa 2 lista ---


## 3. Merge de escenas por fecha

In [4]:
# ============ ETAPA 3/6: MERGE DE ESCENAS POR FECHA (tu 03) ============
from rasterio.merge import merge as rio_merge

EXTS = (".tif", ".tiff", ".jp2", ".TIF", ".JP2")
pattern = re.compile(r".*?(\d{4}_\d{1,2}_\d{1,2}_\d+).*")

groups = defaultdict(list)
for p in COMPOSITE_DIR.iterdir():
    if p.is_file() and p.suffix in EXTS:
        m = pattern.search(p.name)
        if m:
            groups[m.group(1)].append(p)

print(f"Grupos (escenas) detectados: {len(groups)}")

def mosaic_and_save(file_list, out_path):
    srcs = []
    try:
        for fp in file_list:
            srcs.append(rasterio.open(fp))
        mosaic_arr, out_trans = rio_merge(srcs)
        out_meta = srcs[0].meta.copy()
        out_meta.update({"driver": "GTiff", "height": mosaic_arr.shape[1],
                         "width": mosaic_arr.shape[2], "transform": out_trans,
                         "count": mosaic_arr.shape[0]})
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(mosaic_arr)
        print(f"  OK mosaico: {out_path.name} ({mosaic_arr.shape[0]}b, {mosaic_arr.shape[1]}x{mosaic_arr.shape[2]})")
    except Exception as e:
        print(f"  ERROR mosaicing {file_list}: {e}")
    finally:
        for s in srcs:
            try: s.close()
            except: pass

for key, files in groups.items():
    files = sorted(files)
    # [MOD 2026-09-13 HNK] sufijo de la corrida en el nombre del mosaico (se hereda a las teselas)
    out_fp = MOSAIC_DIR / f"{key}{SUFIJO}_mosaic.tif"
    if out_fp.exists():
        continue
    mosaic_and_save(files, out_fp)

print("--- Etapa 3 lista ---")


Grupos (escenas) detectados: 30
  OK mosaico: 2026_6_11_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_12_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_14_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_14_1_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_17_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_19_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_22_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_24_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_24_1_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_27_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_29_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_2_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_2_1_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_4_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_7_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_6_9_0_HNK_mosaic.tif (4b, 10980x10980)
  OK mosaico: 2026_7_12_0_HNK

## 4. Clip a la zona agrícola (shapefile Marcos Juárez)

In [5]:
# ============ ETAPA 4/6: CLIP A ZONA AGRÍCOLA (tu 04) ============
import geopandas as gpd
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping, box

NODATA = np.nan
COMPRESS = "LZW"

gdf_agri = gpd.read_file(SHAPEFILE_AGRI)
if gdf_agri.empty:
    raise SystemExit(f"Shapefile {SHAPEFILE_AGRI} vacío.")
gdf_agri['geometry'] = gdf_agri['geometry'].buffer(0)

def geometries_intersecting_raster(raster_path, gdf_shp):
    with rasterio.open(raster_path) as src:
        rast_crs = src.crs
        bounds = src.bounds
        src_dtype = src.dtypes[0]
    if gdf_shp.crs != rast_crs:
        try:
            gdf_rast = gdf_shp.to_crs(rast_crs)
        except Exception:
            return [], src_dtype
    else:
        gdf_rast = gdf_shp
    bbox_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    try:
        sindex = gdf_rast.sindex
        possible_idx = list(sindex.intersection(bbox_geom.bounds))
        possible = gdf_rast.iloc[possible_idx] if possible_idx else gdf_rast.iloc[[]]
    except Exception:
        possible = gdf_rast
    if len(possible) == 0:
        return [], src_dtype
    intersects = possible[possible.geometry.intersects(bbox_geom)]
    if intersects.empty:
        return [], src_dtype
    return list(intersects.geometry.values), src_dtype

rasters = sorted(glob.glob(str(MOSAIC_DIR / "*.tif")))
print(f"Mosaics a recortar: {len(rasters)}")

for rast_path in tqdm(rasters, desc="Clip"):
    fname = os.path.splitext(os.path.basename(rast_path))[0]
    out_path = CLIP_DIR / f"{fname}_clipped.tif"
    if out_path.exists():
        continue

    geoms, src_dtype = geometries_intersecting_raster(rast_path, gdf_agri)
    if len(geoms) == 0:
        continue
    shapes = [mapping(g) for g in geoms]

    with rasterio.open(rast_path) as src:
        meta = src.meta.copy()
        # fill temporal compatible con el dtype origen; luego pasamos a float32 + NaN
        try:
            info = np.iinfo(np.dtype(src_dtype))
            fill_value = int(info.max)
        except Exception:
            fill_value = 0
        try:
            out_image, out_transform = rio_mask(dataset=src, shapes=shapes, invert=False,
                                                crop=True, nodata=fill_value)
        except Exception as e:
            print(f"  ERROR mask {rast_path}: {e}")
            continue

        out_image = out_image.astype(np.float32)
        out_image[out_image == float(fill_value)] = np.nan

        out_meta = meta.copy()
        out_meta.update({"height": out_image.shape[1], "width": out_image.shape[2],
                         "transform": out_transform, "dtype": "float32",
                         "count": out_image.shape[0], "compress": COMPRESS, "nodata": np.nan})
        try:
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)
        except Exception as e:
            print(f"  ERROR guardando {out_path}: {e}")

print("--- Etapa 4 lista ---")


Mosaics a recortar: 30


Clip: 100%|##########| 30/30 [25:44<00:00, 51.47s/it]


--- Etapa 4 lista ---


## 5. Tiling (1024 px, 20% overlap)

In [6]:
# ============ ETAPA 5/6: TILING 1024px / 20% overlap (tu 05) ============
from rasterio.windows import Window
import warnings
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

def calculate_tile_positions(width, height, tile_size, overlap_percent):
    overlap_pixels = int(tile_size * overlap_percent / 100)
    step_size = tile_size - overlap_pixels
    x_positions, y_positions = [], []
    x = 0
    while x < width:
        x_end = min(x + tile_size, width)
        x_positions.append((x, x_end))
        if x_end >= width: break
        x += step_size
    y = 0
    while y < height:
        y_end = min(y + tile_size, height)
        y_positions.append((y, y_end))
        if y_end >= height: break
        y += step_size
    return x_positions, y_positions

def is_tile_valid(tile_data, threshold_percent=80):
    if tile_data is None or tile_data.size == 0:
        return False
    if np.isinf(tile_data).any():
        return False
    if len(tile_data.shape) == 3:
        valid_pixels = ~np.isnan(tile_data).all(axis=0)
        total_pixels = tile_data.shape[1] * tile_data.shape[2]
        valid_count = valid_pixels.sum()
        if valid_count > 0:
            valid_data = tile_data[:, valid_pixels]
            dark_pixels = np.sum(np.all(valid_data < 50, axis=0))
            dark_percentage_of_valid = (dark_pixels / valid_count) * 100 if valid_count > 0 else 100
        else:
            dark_percentage_of_valid = 100
    else:
        valid_pixels = ~np.isnan(tile_data)
        total_pixels = tile_data.size
        valid_count = valid_pixels.sum()
        dark_percentage_of_valid = (np.sum(tile_data[valid_pixels] < 50) / valid_count * 100) if valid_count > 0 else 100
    valid_percentage = (valid_count / total_pixels) * 100
    return valid_percentage >= 20 and dark_percentage_of_valid < threshold_percent

def calculate_tile_transform(original_transform, x_start, y_start):
    x_coord, y_coord = rasterio.transform.xy(original_transform, y_start, x_start, offset='ul')
    return rasterio.transform.from_origin(x_coord, y_coord, original_transform.a, -original_transform.e)

def generate_tiles_from_mosaic(mosaic_path, output_dir, tile_size, overlap_percent):
    filename = os.path.splitext(os.path.basename(mosaic_path))[0]
    base_name = filename.replace('_mosaic_clipped', '')
    creados = 0
    with rasterio.open(mosaic_path) as src:
        width, height, bands = src.width, src.height, src.count
        x_positions, y_positions = calculate_tile_positions(width, height, tile_size, overlap_percent)
        for row_idx, (y_start, y_end) in enumerate(y_positions):
            for col_idx, (x_start, x_end) in enumerate(x_positions):
                if x_start >= width or y_start >= height:
                    continue
                x_end_adj = min(x_end, width); y_end_adj = min(y_end, height)
                window = Window(x_start, y_start, x_end_adj - x_start, y_end_adj - y_start)
                if window.width <= 0 or window.height <= 0:
                    continue
                try:
                    tile_data = src.read(window=window)
                except Exception:
                    continue
                if not is_tile_valid(tile_data):
                    continue
                if tile_data.shape[1] != tile_size or tile_data.shape[2] != tile_size:
                    padded = np.full((bands, tile_size, tile_size), np.nan, dtype=tile_data.dtype)
                    padded[:, :tile_data.shape[1], :tile_data.shape[2]] = tile_data
                    tile_data = padded
                tile_name = f"{base_name}_tile_r{row_idx:02d}_c{col_idx:02d}.tif"
                tile_path = os.path.join(output_dir, tile_name)
                tile_transform = calculate_tile_transform(src.transform, x_start, y_start)
                tile_meta = src.meta.copy()
                tile_meta.update({'height': tile_data.shape[1], 'width': tile_data.shape[2],
                                  'transform': tile_transform})
                try:
                    with rasterio.open(tile_path, 'w', **tile_meta) as dst:
                        dst.write(tile_data)
                        if src.crs:
                            dst.crs = src.crs
                    creados += 1
                except Exception:
                    continue
    return creados

clipped = [f for f in os.listdir(CLIP_DIR) if f.lower().endswith('.tif')]
# [MOD 2026-09-13] tilear solo mosaicos de MESES_A_PROCESAR (jun/jul ya estaban tileados)
clipped = [f for f in clipped if mes_de(f) in MESES_A_PROCESAR]
print(f"Mosaics recortados a tilear: {len(clipped)}")
total = 0
for fn in tqdm(clipped, desc="Tiling"):
    total += generate_tiles_from_mosaic(str(CLIP_DIR / fn), str(TILES_DIR), TILE_SIZE, OVERLAP_PERCENT)
print(f"Tiles creados: {total}")
print("Guardados en:", TILES_DIR)
print("--- Etapa 5 lista ---")


Mosaics recortados a tilear: 30


Tiling: 100%|##########| 30/30 [26:56<00:00, 53.90s/it]


Tiles creados: 257
Guardados en: D:\Silos\Base_2026_HNK_junjul\procesado_sin_nubes\tiles
--- Etapa 5 lista ---


## 6. Índice INBNV → máscara percentil → polígonos GT

In [7]:
# ============ ETAPA 6/6: INBNV -> MÁSCARA -> POLÍGONOS GT (tu 06) ============
from skimage.filters import threshold_triangle, threshold_otsu
from rasterio.features import shapes as rio_shapes
from scipy.ndimage import label, binary_dilation
from shapely.geometry import shape

# --- 6a. Índice INBNV ---
raster_files = glob.glob(str(TILES_DIR / "*.tif"))
# [MOD 2026-09-13] solo archivos de MESES_A_PROCESAR (jun/jul ya estaban hechos)
raster_files = [f for f in raster_files if mes_de(f) in MESES_A_PROCESAR]
print(f"6a. Calculando INBNV sobre {len(raster_files)} tiles...")
np.seterr(divide='ignore', invalid='ignore')
for file_path in raster_files:
    try:
        with rasterio.open(file_path) as src:
            green = src.read(BANDA_VERDE).astype(np.float32)
            red   = src.read(BANDA_ROJA).astype(np.float32)
            nir   = src.read(BANDA_NIR).astype(np.float32)
            meta = src.meta.copy()
        numerator = (green + red) - nir
        denominator = (green + red) + nir
        inbnv = np.where(denominator == 0, 0, numerator / denominator)
        meta.update({'dtype': 'float32', 'count': 1, 'nodata': -9999})
        inbnv[~np.isfinite(inbnv)] = meta['nodata']
        name_part = os.path.splitext(os.path.basename(file_path))[0]
        out_path = INDEX_DIR / f"{name_part}_INBNV.tif"
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(inbnv, 1)
    except Exception as e:
        print(f"  ERROR INBNV {os.path.basename(file_path)}: {e}")

# --- 6b. Máscara por percentil ---
indice_files = glob.glob(str(INDEX_DIR / "*_INBNV.tif"))
# [MOD 2026-09-13] solo archivos de MESES_A_PROCESAR (jun/jul ya estaban hechos)
indice_files = [f for f in indice_files if mes_de(f) in MESES_A_PROCESAR]
print(f"6b. Umbralizando {len(indice_files)} índices (percentil {PERCENTILE})...")
for path_al_indice in indice_files:
    try:
        with rasterio.open(path_al_indice) as src:
            inbnv = src.read(1); nodata = src.nodata; meta = src.meta.copy()
        valid_pixels = inbnv[inbnv != nodata]
        if valid_pixels.size == 0:
            continue
        threshold = np.percentile(valid_pixels, PERCENTILE)
        binary_mask = np.where(inbnv > threshold, 1, 0).astype(np.uint8)
        if nodata is not None:
            binary_mask[inbnv == nodata] = 255
        meta.update({'dtype': 'uint8', 'nodata': 255})
        base_name = os.path.basename(path_al_indice).replace("_INBNV.tif", "_PERCENTILE_MASK.tif")
        with rasterio.open(MASK_DIR / base_name, 'w', **meta) as dst:
            dst.write(binary_mask, 1)
    except Exception as e:
        print(f"  ERROR máscara {os.path.basename(path_al_indice)}: {e}")

# --- 6c. Máscara -> polígonos (GT semiautomático) ---
mask_files = glob.glob(str(MASK_DIR / "*_MASK.tif"))
# [MOD 2026-09-13] solo archivos de MESES_A_PROCESAR (jun/jul ya estaban hechos)
mask_files = [f for f in mask_files if mes_de(f) in MESES_A_PROCESAR]
print(f"6c. Vectorizando {len(mask_files)} máscaras a polígonos...")
for mask_path in mask_files:
    try:
        with rasterio.open(mask_path) as src:
            mask_arr = src.read(1); transform = src.transform; crs = src.crs
        binary = (mask_arr == 1).astype(np.uint8)
        labeled, num_features = label(binary)
        cleaned = np.zeros_like(binary)
        for i in range(1, num_features + 1):
            region = (labeled == i)
            if region.sum() > MIN_PIXELS:
                cleaned[region] = 1
        buffered = binary_dilation(cleaned, iterations=BUFFER_PIXELS).astype(np.uint8)
        polygons = [shape(geom) for geom, val in
                    rio_shapes(buffered, mask=buffered.astype(bool), transform=transform) if val == 1]
        if not polygons:
            continue
        gdf = gpd.GeoDataFrame(geometry=polygons, crs=crs)
        out_file = POLY_DIR / os.path.basename(mask_path).replace("_MASK.tif", "_POLYGONS.shp")
        gdf.to_file(out_file)
    except Exception as e:
        print(f"  ERROR polígonos {os.path.basename(mask_path)}: {e}")

print("\n=== PIPELINE 2026 COMPLETO ===")
print("Tiles RGB+NIR:", TILES_DIR)
print("Polígonos GT (a limpiar en QGIS):", POLY_DIR)


6a. Calculando INBNV sobre 257 tiles...
6b. Umbralizando 257 índices (percentil 99.8)...
6c. Vectorizando 257 máscaras a polígonos...

=== PIPELINE 2026 COMPLETO ===
Tiles RGB+NIR: D:\Silos\Base_2026_HNK_junjul\procesado_sin_nubes\tiles
Polígonos GT (a limpiar en QGIS): D:\Silos\Base_2026_HNK_junjul\procesado_sin_nubes\tiles\indices_INBNV\masks\polygons


## 7. Selección de 2 teselas 20HNK para el test 2026

[MOD 2026-09-13 HNK] Celda nueva. 1 tesela de junio + 1 de julio, sorteadas (semilla 456 + mes, mismo criterio que agosto) entre las teselas que cumplen: 100% al norte de la cobertura del jun/jul original (zona que solo cubre 20HNK), ≥70% de píxeles válidos y sin nubes enmascaradas. No se reemplaza ninguna a ojo.


In [8]:
# [MOD 2026-09-13 HNK] NUEVA celda: seleccion de 2 teselas 20HNK (1 junio + 1 julio) para el test 2026
import random, shutil

TEST_DIR   = r"D:\Silos\Base_2026\test"
REF_MOSAIC = r"D:\Silos\Base_2026\procesado_sin_nubes\mosaics\2026_6_22_0_mosaic.tif"  # mosaico jun/jul original (HNH+HNJ)
MIN_VALIDOS, MAX_NUBE = 0.70, 0.01

with rasterio.open(REF_MOSAIC) as src:
    NORTE_JUNJUL = src.bounds.top            # borde norte de la cobertura jun/jul original

def calidad(path):
    with rasterio.open(path) as src:
        a = src.read([1, 2, 3]); bottom = src.bounds.bottom
    nan  = np.isnan(a).all(axis=0)
    nube = (np.nan_to_num(a) == 0).all(axis=0) & ~nan     # pixeles enmascarados por SCL
    return bottom, 1 - nan.mean(), nube.mean()

def copy_shp(stem, src, dst):
    n = 0
    for ext in (".shp", ".shx", ".dbf", ".prj", ".cpg"):
        p = os.path.join(src, stem + ext)
        if os.path.exists(p) and not os.path.exists(os.path.join(dst, stem + ext)): shutil.copy2(p, dst)
        if os.path.exists(p): n += 1
    return n

for mes in (6, 7):
    tifs = sorted(f for f in os.listdir(TILES_DIR) if f.endswith(".tif") and mes_de(f) == mes)
    pool = []
    for f in tifs:
        bottom, validos, nube = calidad(os.path.join(TILES_DIR, f))
        if bottom >= NORTE_JUNJUL and validos >= MIN_VALIDOS and nube <= MAX_NUBE:
            pool.append(f)
    rng = random.Random(456 + mes)
    rng.shuffle(pool)
    print(f"Mes {mes}: {len(tifs)} teselas 20HNK | {len(pool)} cumplen criterios | orden sorteado (5 primeras): {pool[:5]}")
    if not pool:
        continue
    t = pool[0]
    stem = os.path.splitext(t)[0]
    shutil.copy2(os.path.join(TILES_DIR, t), TEST_DIR)
    ncand = copy_shp(stem + "_PERCENTILE_POLYGONS", str(POLY_DIR), TEST_DIR)
    print(f"  -> seleccionada: {t} ({'con candidatos' if ncand else 'SIN candidatos (digitalizar a mano si hay bolsas)'})")
print(f"\nCopiado a {TEST_DIR}. Curá las nuevas en QGIS antes de validar.")


Mes 6: 196 teselas 20HNK | 17 cumplen criterios | orden sorteado (5 primeras): ['2026_6_19_0_HNK_tile_r05_c04.tif', '2026_6_14_1_HNK_tile_r05_c05.tif', '2026_6_19_0_HNK_tile_r05_c06.tif', '2026_6_19_0_HNK_tile_r05_c05.tif', '2026_6_22_0_HNK_tile_r05_c06.tif']
  -> seleccionada: 2026_6_19_0_HNK_tile_r05_c04.tif (con candidatos)
Mes 7: 61 teselas 20HNK | 5 cumplen criterios | orden sorteado (5 primeras): ['2026_7_9_0_HNK_tile_r05_c06.tif', '2026_7_7_0_HNK_tile_r05_c05.tif', '2026_7_9_0_HNK_tile_r05_c04.tif', '2026_7_9_0_HNK_tile_r05_c05.tif', '2026_7_7_0_HNK_tile_r05_c06.tif']
  -> seleccionada: 2026_7_9_0_HNK_tile_r05_c06.tif (con candidatos)

Copiado a D:\Silos\Base_2026\test. Curá las nuevas en QGIS antes de validar.
